# Week 3: Contrastive Probe Training

## The Key Insight

When the model has high entropy, we need to check **WHAT it's uncertain about**:

### Example: "The authentication is"

**Model's top candidates might be:**
1. ` Firebase` (probability 0.15) → CODE
2. ` JWT` (probability 0.12) → CODE
3. ` a` (probability 0.11) → LANGUAGE
4. ` done` (probability 0.10) → LANGUAGE
5. ` OAuth` (probability 0.09) → CODE

**High entropy because:**
- If uncertain between Firebase/JWT/OAuth → **CODE uncertainty** (stop and ask!)
- If uncertain between a/done/handled → **LANGUAGE uncertainty** (continue)

## The Solution: Contrastive Training

1. **Training:** For each prompt, generate candidate next tokens and label:
   - "The authentication is **Firebase**" → CODE (1)
   - "The authentication is **a**" → LANGUAGE (0)

2. **Testing:** When entropy is high:
   - Get top-K candidate next tokens
   - Classify each "prompt + candidate"
   - If majority are CODE → stop
   - If majority are LANGUAGE → continue

---

In [1]:
# Cell 1: Install
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn

In [2]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import entropy as scipy_entropy
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.notebook import tqdm
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print("✅ Imports")

✅ Imports


In [3]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    output_hidden_states=True
)
model.eval()

print(f"✅ Model loaded on {model.device}")

Loading codellama/CodeLlama-7b-Instruct-hf...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Model loaded on cuda:0


# Part 1: Base Training Examples

In [4]:
# Cell 4: Comprehensive and diverse base prompts with candidate continuations

# ============================================================================
# CODE UNCERTAINTY: Prompts where model is uncertain about code-specific tokens
# ============================================================================

CODE_BASE_PROMPTS = [
    # ===== AUTHENTICATION & SECURITY =====
    {
        'prompt': 'The authentication is done using',
        'code_continuations': [' JWT', ' OAuth2', ' OAuth', ' Firebase', ' Auth0', ' Passport', ' Keycloak', ' Okta'],
        'lang_continuations': [' a', ' an', ' the', ' some', ' several', ' our'],
        'desc': 'Auth mechanism'
    },
    {
        'prompt': 'Password hashing uses',
        'code_continuations': [' bcrypt', ' argon2', ' scrypt', ' PBKDF2', ' SHA256', ' Argon2id'],
        'lang_continuations': [' a', ' the', ' our', ' standard', ' secure'],
        'desc': 'Hashing algorithm'
    },
    {
        'prompt': 'Tokens are generated with',
        'code_continuations': [' jwt', ' uuid', ' nanoid', ' crypto', ' jsonwebtoken', ' jose'],
        'lang_continuations': [' a', ' the', ' our', ' secure'],
        'desc': 'Token library'
    },
    {
        'prompt': 'OAuth is implemented via',
        'code_continuations': [' Passport', ' Auth0', ' Firebase', ' NextAuth', ' Keycloak'],
        'lang_continuations': [' a', ' the', ' our', ' standard'],
        'desc': 'OAuth provider'
    },
    {
        'prompt': 'Session storage uses',
        'code_continuations': [' Redis', ' Memcached', ' MongoDB', ' PostgreSQL', ' express-session'],
        'lang_continuations': [' a', ' the', ' our', ' secure'],
        'desc': 'Session store'
    },

    # ===== WEB FRAMEWORKS & BACKEND =====
    {
        'prompt': 'The API is built with',
        'code_continuations': [' Express', ' FastAPI', ' Flask', ' Django', ' NestJS', ' Koa', ' Fastify'],
        'lang_continuations': [' a', ' the', ' our', ' modern'],
        'desc': 'API framework'
    },
    {
        'prompt': 'The backend uses',
        'code_continuations': [' Node.js', ' Python', ' Go', ' Rust', ' Java', ' Ruby', ' PHP'],
        'lang_continuations': [' a', ' the', ' our', ' modern'],
        'desc': 'Backend language'
    },
    {
        'prompt': 'We use the',
        'code_continuations': [' Express', ' FastAPI', ' Gin', ' Axum', ' Spring', ' Rails'],
        'lang_continuations': [' standard', ' default', ' recommended', ' latest'],
        'desc': 'Framework choice'
    },
    {
        'prompt': 'GraphQL is implemented with',
        'code_continuations': [' Apollo', ' GraphQL-Yoga', ' Hasura', ' Prisma', ' Relay'],
        'lang_continuations': [' a', ' the', ' our', ' custom'],
        'desc': 'GraphQL server'
    },
    {
        'prompt': 'The REST API uses',
        'code_continuations': [' Express', ' FastAPI', ' Actix', ' Gin', ' Jersey'],
        'lang_continuations': [' a', ' the', ' standard', ' RESTful'],
        'desc': 'REST framework'
    },

    # ===== FRONTEND FRAMEWORKS =====
    {
        'prompt': 'The frontend uses',
        'code_continuations': [' React', ' Vue', ' Angular', ' Svelte', ' Next.js', ' Nuxt', ' SolidJS'],
        'lang_continuations': [' a', ' the', ' our', ' modern'],
        'desc': 'Frontend framework'
    },
    {
        'prompt': 'State management is',
        'code_continuations': [' Redux', ' Zustand', ' Pinia', ' MobX', ' Recoil', ' Jotai', ' XState'],
        'lang_continuations': [' a', ' the', ' handled', ' done'],
        'desc': 'State library'
    },
    {
        'prompt': 'The component library is',
        'code_continuations': [' Material-UI', ' Chakra', ' Ant Design', ' Shadcn', ' Tailwind', ' Bootstrap'],
        'lang_continuations': [' a', ' our', ' custom', ' based'],
        'desc': 'UI library'
    },
    {
        'prompt': 'Routing is handled by',
        'code_continuations': [' React Router', ' Next.js', ' Vue Router', ' TanStack Router'],
        'lang_continuations': [' a', ' the', ' our', ' custom'],
        'desc': 'Router library'
    },

    # ===== DATABASES =====
    {
        'prompt': 'The database is',
        'code_continuations': [' PostgreSQL', ' MongoDB', ' MySQL', ' Redis', ' SQLite', ' Cassandra', ' DynamoDB'],
        'lang_continuations': [' a', ' the', ' our', ' located'],
        'desc': 'Database type'
    },
    {
        'prompt': 'The ORM is',
        'code_continuations': [' Prisma', ' TypeORM', ' Sequelize', ' SQLAlchemy', ' Django ORM', ' Hibernate'],
        'lang_continuations': [' a', ' the', ' our', ' configured'],
        'desc': 'ORM choice'
    },
    {
        'prompt': 'We use model.',
        'code_continuations': ['findMany', 'findUnique', 'create', 'update', 'delete', 'upsert'],
        'lang_continuations': [' to', ' for', ' in', ' as'],
        'desc': 'ORM method'
    },
    {
        'prompt': 'Database migrations use',
        'code_continuations': [' Prisma', ' Alembic', ' Flyway', ' Knex', ' TypeORM', ' Liquibase'],
        'lang_continuations': [' a', ' the', ' our', ' automated'],
        'desc': 'Migration tool'
    },
    {
        'prompt': 'SELECT password FROM',
        'code_continuations': [' users', ' accounts', ' credentials', ' auth_table', ' user_profiles'],
        'lang_continuations': [' the', ' a', ' our'],
        'desc': 'SQL table name'
    },
    {
        'prompt': 'The cache layer is',
        'code_continuations': [' Redis', ' Memcached', ' Varnish', ' Cloudflare', ' KeyDB'],
        'lang_continuations': [' a', ' the', ' implemented', ' using'],
        'desc': 'Cache system'
    },

    # ===== MACHINE LEARNING & DATA SCIENCE =====
    {
        'prompt': 'We import',
        'code_continuations': [' numpy', ' pandas', ' torch', ' tensorflow', ' scikit-learn', ' transformers'],
        'lang_continuations': [' the', ' a', ' several', ' multiple'],
        'desc': 'Library import'
    },
    {
        'prompt': 'The model architecture is',
        'code_continuations': [' BERT', ' GPT', ' ResNet', ' Transformer', ' LSTM', ' CNN', ' VGG'],
        'lang_continuations': [' a', ' based', ' designed', ' complex'],
        'desc': 'ML architecture'
    },
    {
        'prompt': 'Training uses optimizer =',
        'code_continuations': [' Adam', ' AdamW', ' SGD', ' RMSprop', ' Adagrad'],
        'lang_continuations': [' a', ' the', ' our', ' custom'],
        'desc': 'Optimizer type'
    },
    {
        'prompt': 'Data preprocessing uses',
        'code_continuations': [' pandas', ' sklearn', ' numpy', ' polars', ' dask'],
        'lang_continuations': [' a', ' the', ' several', ' multiple'],
        'desc': 'Preprocessing library'
    },
    {
        'prompt': 'The deep learning framework is',
        'code_continuations': [' PyTorch', ' TensorFlow', ' JAX', ' Keras', ' MXNet'],
        'lang_continuations': [' a', ' the', ' based', ' using'],
        'desc': 'DL framework'
    },

    # ===== CLOUD & DEVOPS =====
    {
        'prompt': 'Deployment is on',
        'code_continuations': [' AWS', ' GCP', ' Azure', ' Vercel', ' Netlify', ' Heroku', ' DigitalOcean'],
        'lang_continuations': [' a', ' the', ' our', ' production'],
        'desc': 'Cloud provider'
    },
    {
        'prompt': 'The container runtime is',
        'code_continuations': [' Docker', ' containerd', ' CRI-O', ' Podman'],
        'lang_continuations': [' a', ' the', ' our', ' configured'],
        'desc': 'Container runtime'
    },
    {
        'prompt': 'Orchestration uses',
        'code_continuations': [' Kubernetes', ' Docker Swarm', ' Nomad', ' ECS', ' EKS'],
        'lang_continuations': [' a', ' the', ' our', ' complex'],
        'desc': 'Orchestration tool'
    },
    {
        'prompt': 'CI/CD is',
        'code_continuations': [' GitHub Actions', ' GitLab CI', ' Jenkins', ' CircleCI', ' Travis'],
        'lang_continuations': [' a', ' automated', ' configured', ' set'],
        'desc': 'CI/CD platform'
    },
    {
        'prompt': 'Infrastructure as Code uses',
        'code_continuations': [' Terraform', ' Pulumi', ' CloudFormation', ' CDK', ' Ansible'],
        'lang_continuations': [' a', ' the', ' our', ' modern'],
        'desc': 'IaC tool'
    },

    # ===== TESTING =====
    {
        'prompt': 'Unit tests use',
        'code_continuations': [' Jest', ' Vitest', ' pytest', ' unittest', ' JUnit', ' Mocha'],
        'lang_continuations': [' a', ' the', ' our', ' standard'],
        'desc': 'Test framework'
    },
    {
        'prompt': 'E2E testing uses',
        'code_continuations': [' Playwright', ' Cypress', ' Selenium', ' Puppeteer', ' TestCafe'],
        'lang_continuations': [' a', ' the', ' our', ' automated'],
        'desc': 'E2E framework'
    },
    {
        'prompt': 'API testing is done with',
        'code_continuations': [' Postman', ' Insomnia', ' Hoppscotch', ' REST Client', ' Thunder Client'],
        'lang_continuations': [' a', ' the', ' our', ' manual'],
        'desc': 'API test tool'
    },

    # ===== PACKAGE MANAGERS & BUILD TOOLS =====
    {
        'prompt': 'Package management uses',
        'code_continuations': [' npm', ' yarn', ' pnpm', ' bun', ' pip', ' poetry', ' cargo'],
        'lang_continuations': [' a', ' the', ' our', ' standard'],
        'desc': 'Package manager'
    },
    {
        'prompt': 'The bundler is',
        'code_continuations': [' Vite', ' Webpack', ' Rollup', ' esbuild', ' Parcel', ' Turbopack'],
        'lang_continuations': [' a', ' the', ' configured', ' using'],
        'desc': 'Build tool'
    },
    {
        'prompt': 'Monorepo management uses',
        'code_continuations': [' Turborepo', ' Nx', ' Lerna', ' Rush', ' Bazel'],
        'lang_continuations': [' a', ' the', ' our', ' complex'],
        'desc': 'Monorepo tool'
    },

    # ===== LOGGING & MONITORING =====
    {
        'prompt': 'Logging is done with',
        'code_continuations': [' Winston', ' Pino', ' Bunyan', ' Log4j', ' Logrus', ' slog'],
        'lang_continuations': [' a', ' the', ' our', ' structured'],
        'desc': 'Logging library'
    },
    {
        'prompt': 'Monitoring uses',
        'code_continuations': [' Prometheus', ' Grafana', ' DataDog', ' New Relic', ' Sentry'],
        'lang_continuations': [' a', ' the', ' our', ' comprehensive'],
        'desc': 'Monitoring tool'
    },
    {
        'prompt': 'Error tracking is',
        'code_continuations': [' Sentry', ' Rollbar', ' Bugsnag', ' Airbrake', ' LogRocket'],
        'lang_continuations': [' done', ' handled', ' implemented', ' configured'],
        'desc': 'Error tracking'
    },

    # ===== MESSAGING & QUEUES =====
    {
        'prompt': 'Message queue is',
        'code_continuations': [' RabbitMQ', ' Kafka', ' Redis', ' NATS', ' AWS SQS', ' Pulsar'],
        'lang_continuations': [' a', ' the', ' implemented', ' using'],
        'desc': 'Message queue'
    },
    {
        'prompt': 'Real-time communication uses',
        'code_continuations': [' Socket.io', ' WebSocket', ' Pusher', ' Ably', ' SignalR'],
        'lang_continuations': [' a', ' the', ' our', ' bidirectional'],
        'desc': 'Real-time library'
    },

    # ===== VALIDATION & PARSING =====
    {
        'prompt': 'Input validation uses',
        'code_continuations': [' Zod', ' Yup', ' Joi', ' Valibot', ' AJV', ' class-validator'],
        'lang_continuations': [' a', ' the', ' strict', ' comprehensive'],
        'desc': 'Validation library'
    },
    {
        'prompt': 'Date handling uses',
        'code_continuations': [' date-fns', ' dayjs', ' luxon', ' moment', ' date-time'],
        'lang_continuations': [' a', ' the', ' our', ' custom'],
        'desc': 'Date library'
    },

    # ===== API CLIENTS & HTTP =====
    {
        'prompt': 'HTTP requests use',
        'code_continuations': [' axios', ' fetch', ' ky', ' got', ' requests', ' urllib'],
        'lang_continuations': [' a', ' the', ' native', ' built-in'],
        'desc': 'HTTP client'
    },
    {
        'prompt': 'API client generation uses',
        'code_continuations': [' OpenAPI', ' tRPC', ' GraphQL Codegen', ' Swagger'],
        'lang_continuations': [' a', ' the', ' automated', ' type-safe'],
        'desc': 'API codegen'
    },
]

# ============================================================================
# LANGUAGE UNCERTAINTY: Prompts with only generic word uncertainty
# ============================================================================

LANGUAGE_BASE_PROMPTS = [
    # ===== CONNECTORS & PASSIVE VOICE (The main problem!) =====
    {
        'prompt': 'The authentication is',
        'code_continuations': [' JWT', ' Firebase', ' OAuth'],  # Wrong context
        'lang_continuations': [' a', ' an', ' done', ' handled', ' implemented', ' performed', ' required', ' critical'],
        'desc': 'Connector "is"'
    },
    {
        'prompt': 'Verification is done',
        'code_continuations': [],
        'lang_continuations': [' by', ' through', ' using', ' via', ' with', ' before', ' after'],
        'desc': 'Connector "done"'
    },
    {
        'prompt': 'The user id is',
        'code_continuations': [],
        'lang_continuations': [' a', ' an', ' the', ' stored', ' saved', ' retrieved', ' generated', ' unique'],
        'desc': 'Connector "is"'
    },
    {
        'prompt': 'Processing is handled',
        'code_continuations': [],
        'lang_continuations': [' by', ' through', ' using', ' asynchronously', ' efficiently'],
        'desc': 'Connector "handled"'
    },
    {
        'prompt': 'Data is stored',
        'code_continuations': [],
        'lang_continuations': [' in', ' on', ' within', ' securely', ' persistently', ' temporarily'],
        'desc': 'Passive "stored"'
    },
    {
        'prompt': 'Tokens are validated',
        'code_continuations': [],
        'lang_continuations': [' by', ' using', ' through', ' before', ' after', ' to'],
        'desc': 'Passive "validated"'
    },
    {
        'prompt': 'The session is created',
        'code_continuations': [],
        'lang_continuations': [' when', ' after', ' before', ' upon', ' during'],
        'desc': 'Passive "created"'
    },
    {
        'prompt': 'Passwords are hashed',
        'code_continuations': [],
        'lang_continuations': [' before', ' using', ' with', ' to', ' for'],
        'desc': 'Passive "hashed"'
    },
    {
        'prompt': 'The request is',
        'code_continuations': [],
        'lang_continuations': [' sent', ' made', ' processed', ' handled', ' validated', ' authenticated'],
        'desc': 'Connector "is"'
    },

    # ===== EXPLANATORY TEXT =====
    {
        'prompt': 'The code works by',
        'code_continuations': [],
        'lang_continuations': [' using', ' calling', ' implementing', ' executing', ' first', ' iterating'],
        'desc': 'Process explanation'
    },
    {
        'prompt': 'The function is responsible for',
        'code_continuations': [],
        'lang_continuations': [' handling', ' managing', ' processing', ' validating', ' checking', ' ensuring'],
        'desc': 'Responsibility'
    },
    {
        'prompt': 'This approach',
        'code_continuations': [],
        'lang_continuations': [' is', ' allows', ' ensures', ' provides', ' enables', ' helps'],
        'desc': 'Approach description'
    },
    {
        'prompt': 'The main purpose is to',
        'code_continuations': [],
        'lang_continuations': [' ensure', ' provide', ' handle', ' manage', ' validate', ' process'],
        'desc': 'Purpose statement'
    },
    {
        'prompt': 'The algorithm works by',
        'code_continuations': [],
        'lang_continuations': [' first', ' iterating', ' comparing', ' sorting', ' filtering', ' mapping'],
        'desc': 'Algorithm explanation'
    },
    {
        'prompt': 'The system',
        'code_continuations': [],
        'lang_continuations': [' uses', ' handles', ' processes', ' manages', ' ensures', ' provides'],
        'desc': 'System description'
    },
    {
        'prompt': 'The authentication system',
        'code_continuations': [],
        'lang_continuations': [' ensures', ' provides', ' handles', ' manages', ' validates', ' verifies'],
        'desc': 'System description'
    },

    # ===== INSTRUCTIONS =====
    {
        'prompt': 'To implement authentication,',
        'code_continuations': [],
        'lang_continuations': [' first', ' you', ' we', ' start', ' begin', ' ensure'],
        'desc': 'Instruction'
    },
    {
        'prompt': 'First, you need to',
        'code_continuations': [],
        'lang_continuations': [' install', ' import', ' create', ' configure', ' set', ' ensure'],
        'desc': 'Sequential instruction'
    },
    {
        'prompt': 'Make sure to',
        'code_continuations': [],
        'lang_continuations': [' install', ' configure', ' set', ' check', ' verify', ' validate'],
        'desc': 'Imperative instruction'
    },
    {
        'prompt': 'Before running the code,',
        'code_continuations': [],
        'lang_continuations': [' ensure', ' make', ' check', ' verify', ' install', ' configure'],
        'desc': 'Precondition'
    },
    {
        'prompt': 'After installation,',
        'code_continuations': [],
        'lang_continuations': [' you', ' we', ' the', ' run', ' configure', ' restart'],
        'desc': 'Next step'
    },

    # ===== COMPARISONS =====
    {
        'prompt': 'Unlike other methods,',
        'code_continuations': [],
        'lang_continuations': [' this', ' our', ' the', ' it', ' we'],
        'desc': 'Contrast'
    },
    {
        'prompt': 'Compared to REST,',
        'code_continuations': [],
        'lang_continuations': [' GraphQL', ' this', ' our', ' the', ' it'],
        'desc': 'Comparison'
    },
    {
        'prompt': 'The difference is',
        'code_continuations': [],
        'lang_continuations': [' that', ' in', ' how', ' the', ' subtle'],
        'desc': 'Difference explanation'
    },
    {
        'prompt': 'Similar to',
        'code_continuations': [],
        'lang_continuations': [' the', ' other', ' previous', ' traditional', ' conventional'],
        'desc': 'Similarity'
    },

    # ===== QUESTIONS =====
    {
        'prompt': 'How does the authentication',
        'code_continuations': [],
        'lang_continuations': [' work', ' flow', ' process', ' system', ' mechanism'],
        'desc': 'How question'
    },
    {
        'prompt': 'What are the benefits of',
        'code_continuations': [],
        'lang_continuations': [' using', ' this', ' the', ' our', ' that'],
        'desc': 'What question'
    },
    {
        'prompt': 'Why is this',
        'code_continuations': [],
        'lang_continuations': [' approach', ' method', ' important', ' necessary', ' recommended', ' preferred'],
        'desc': 'Why question'
    },
    {
        'prompt': 'When should you',
        'code_continuations': [],
        'lang_continuations': [' use', ' implement', ' consider', ' apply', ' choose'],
        'desc': 'When question'
    },

    # ===== COMMENTS =====
    {
        'prompt': '# TODO: Fix the',
        'code_continuations': [],
        'lang_continuations': [' bug', ' issue', ' error', ' problem', ' authentication'],
        'desc': 'TODO comment'
    },
    {
        'prompt': '// This function handles',
        'code_continuations': [],
        'lang_continuations': [' the', ' user', ' authentication', ' data', ' request'],
        'desc': 'JS comment'
    },
    {
        'prompt': '# Note: The authentication',
        'code_continuations': [],
        'lang_continuations': [' is', ' must', ' should', ' requires', ' flow'],
        'desc': 'Note comment'
    },
    {
        'prompt': '/* Important:',
        'code_continuations': [],
        'lang_continuations': [' Make', ' Ensure', ' This', ' The', ' Always'],
        'desc': 'Important comment'
    },

    # ===== CONDITIONAL & TEMPORAL =====
    {
        'prompt': 'If the authentication fails,',
        'code_continuations': [],
        'lang_continuations': [' the', ' we', ' return', ' throw', ' redirect'],
        'desc': 'Conditional'
    },
    {
        'prompt': 'When the user logs in,',
        'code_continuations': [],
        'lang_continuations': [' the', ' we', ' a', ' their', ' create'],
        'desc': 'Temporal'
    },
    {
        'prompt': 'After validation,',
        'code_continuations': [],
        'lang_continuations': [' the', ' we', ' data', ' proceed', ' continue'],
        'desc': 'Sequential'
    },

    # ===== TECHNICAL WRITING =====
    {
        'prompt': 'In summary,',
        'code_continuations': [],
        'lang_continuations': [' the', ' this', ' our', ' we', ' authentication'],
        'desc': 'Summary'
    },
    {
        'prompt': 'As mentioned earlier,',
        'code_continuations': [],
        'lang_continuations': [' the', ' this', ' we', ' our', ' authentication'],
        'desc': 'Reference back'
    },
    {
        'prompt': 'It is important to',
        'code_continuations': [],
        'lang_continuations': [' note', ' understand', ' remember', ' ensure', ' verify'],
        'desc': 'Emphasis'
    },
    {
        'prompt': 'The recommended way to',
        'code_continuations': [],
        'lang_continuations': [' implement', ' handle', ' manage', ' do', ' approach'],
        'desc': 'Recommendation'
    },
]

print(f"\n{'='*80}")
print(f"COMPREHENSIVE TRAINING DATA")
print(f"{'='*80}")
print(f"Code base prompts: {len(CODE_BASE_PROMPTS)}")
print(f"Language base prompts: {len(LANGUAGE_BASE_PROMPTS)}")

# Calculate expected training examples
code_examples = sum(len(p['code_continuations']) + len(p['lang_continuations']) for p in CODE_BASE_PROMPTS)
lang_examples = sum(len(p['lang_continuations']) + len(p.get('code_continuations', [])) for p in LANGUAGE_BASE_PROMPTS)
print(f"\nExpected training examples: ~{code_examples + lang_examples}")
print(f"  From CODE prompts: ~{code_examples}")
print(f"  From LANGUAGE prompts: ~{lang_examples}")
print(f"{'='*80}")


COMPREHENSIVE TRAINING DATA
Code base prompts: 45
Language base prompts: 40

Expected training examples: ~664
  From CODE prompts: ~437
  From LANGUAGE prompts: ~227


# Part 2: Generate Contrastive Training Data

In [5]:
# Cell 5: Generate contrastive training examples

TRAIN_EXAMPLES = []

# From CODE prompts: prompt + code_continuation → CODE (1)
for prompt_data in CODE_BASE_PROMPTS:
    prompt = prompt_data['prompt']
    for cont in prompt_data['code_continuations']:
        TRAIN_EXAMPLES.append({
            'text': prompt + cont,
            'label': 1,
            'type': 'code_uncertainty',
            'base_prompt': prompt,
            'continuation': cont,
            'desc': prompt_data['desc']
        })

    # prompt + lang_continuation → LANGUAGE (0)
    for cont in prompt_data['lang_continuations']:
        TRAIN_EXAMPLES.append({
            'text': prompt + cont,
            'label': 0,
            'type': 'language_uncertainty',
            'base_prompt': prompt,
            'continuation': cont,
            'desc': prompt_data['desc'] + ' (lang)'
        })

# From LANGUAGE prompts: prompt + lang_continuation → LANGUAGE (0)
for prompt_data in LANGUAGE_BASE_PROMPTS:
    prompt = prompt_data['prompt']
    for cont in prompt_data['lang_continuations']:
        TRAIN_EXAMPLES.append({
            'text': prompt + cont,
            'label': 0,
            'type': 'language_uncertainty',
            'base_prompt': prompt,
            'continuation': cont,
            'desc': prompt_data['desc']
        })

    # If there are code continuations (wrong context), mark as CODE
    for cont in prompt_data.get('code_continuations', []):
        TRAIN_EXAMPLES.append({
            'text': prompt + cont,
            'label': 1,
            'type': 'code_uncertainty',
            'base_prompt': prompt,
            'continuation': cont,
            'desc': prompt_data['desc'] + ' (wrong code)'
        })

print(f"\n{'='*80}")
print(f"CONTRASTIVE TRAINING DATASET")
print(f"{'='*80}")
print(f"\nTotal examples: {len(TRAIN_EXAMPLES)}")
print(f"  Code uncertainty (label=1): {sum(1 for e in TRAIN_EXAMPLES if e['label'] == 1)}")
print(f"  Language uncertainty (label=0): {sum(1 for e in TRAIN_EXAMPLES if e['label'] == 0)}")
print(f"\nKey distinction:")
print(f"  Prompt + CODE token (Firebase, JWT, bcrypt) → CODE (1)")
print(f"  Prompt + LANGUAGE token (a, done, handled) → LANGUAGE (0)")
print(f"\n{'='*80}")

# Show examples
print(f"\nExample CODE training instances:")
for ex in [e for e in TRAIN_EXAMPLES if e['label'] == 1][:5]:
    print(f"  '{ex['text']}' → CODE")

print(f"\nExample LANGUAGE training instances:")
for ex in [e for e in TRAIN_EXAMPLES if e['label'] == 0][:5]:
    print(f"  '{ex['text']}' → LANGUAGE")


CONTRASTIVE TRAINING DATASET

Total examples: 664
  Code uncertainty (label=1): 258
  Language uncertainty (label=0): 406

Key distinction:
  Prompt + CODE token (Firebase, JWT, bcrypt) → CODE (1)
  Prompt + LANGUAGE token (a, done, handled) → LANGUAGE (0)


Example CODE training instances:
  'The authentication is done using JWT' → CODE
  'The authentication is done using OAuth2' → CODE
  'The authentication is done using OAuth' → CODE
  'The authentication is done using Firebase' → CODE
  'The authentication is done using Auth0' → CODE

Example LANGUAGE training instances:
  'The authentication is done using a' → LANGUAGE
  'The authentication is done using an' → LANGUAGE
  'The authentication is done using the' → LANGUAGE
  'The authentication is done using some' → LANGUAGE
  'The authentication is done using several' → LANGUAGE


# Part 3: Train Contrastive Probe

In [6]:
# Cell 6: Extract hidden states

SELECTED_LAYERS = [8, 16, 31]

def get_multi_layer_state(text: str, layers: List[int]) -> np.ndarray:
    """Extract and concatenate hidden states from multiple layers."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    states = [
        outputs.hidden_states[layer_idx + 1][:, -1, :].cpu().numpy()[0]
        for layer_idx in layers
    ]
    return np.concatenate(states).astype(np.float32)

print(f"Extracting hidden states from layers {SELECTED_LAYERS}...")
X_train = []
y_train = []

for example in tqdm(TRAIN_EXAMPLES, desc="Extracting"):
    h = get_multi_layer_state(example['text'], SELECTED_LAYERS)
    X_train.append(h)
    y_train.append(example['label'])

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"\n✅ Hidden states: {X_train.shape}")
print(f"   Feature dimension: {X_train.shape[1]:,} (3 layers × 4096)")

Extracting hidden states from layers [8, 16, 31]...


Extracting:   0%|          | 0/664 [00:00<?, ?it/s]


✅ Hidden states: (664, 12288)
   Feature dimension: 12,288 (3 layers × 4096)


In [7]:
# Cell 7: Train classifier

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

probe = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(probe, X_train_scaled, y_train, cv=cv)

cv_accuracy = accuracy_score(y_train, y_pred_cv)

print(f"\n{'='*80}")
print(f"CONTRASTIVE PROBE PERFORMANCE")
print(f"{'='*80}")
print(f"\nLayers: {SELECTED_LAYERS}")
print(f"Training examples: {len(y_train)}")
print(f"5-Fold CV Accuracy: {cv_accuracy:.1%}")

print(f"\n{classification_report(y_train, y_pred_cv, target_names=['Language Token', 'Code Token'])}")

# Confusion matrix
cm = confusion_matrix(y_train, y_pred_cv)
print(f"Confusion Matrix:")
print(f"                Pred LANG  Pred CODE")
print(f"True LANG          {cm[0,0]:>4}       {cm[0,1]:>4}")
print(f"True CODE          {cm[1,0]:>4}       {cm[1,1]:>4}")

# Train final model
probe.fit(X_train_scaled, y_train)
print(f"\n✅ Contrastive probe trained")


CONTRASTIVE PROBE PERFORMANCE

Layers: [8, 16, 31]
Training examples: 664
5-Fold CV Accuracy: 99.1%

                precision    recall  f1-score   support

Language Token       1.00      0.99      0.99       406
    Code Token       0.98      1.00      0.99       258

      accuracy                           0.99       664
     macro avg       0.99      0.99      0.99       664
  weighted avg       0.99      0.99      0.99       664

Confusion Matrix:
                Pred LANG  Pred CODE
True LANG           401          5
True CODE             1        257

✅ Contrastive probe trained


# Part 4: Contrastive Generation

In [8]:
# Cell 8: Helper functions

def softmax(logits: np.ndarray) -> np.ndarray:
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    return exp_logits / np.sum(exp_logits)

def entropy_from_probs(probs: np.ndarray) -> float:
    if len(probs) == 0 or np.sum(probs) == 0:
        return 0.0
    norm_probs = probs / np.sum(probs)
    return float(scipy_entropy(norm_probs, base=2))

def classify_token_type(text: str) -> Tuple[int, float]:
    """Classify: 1=code token, 0=language token."""
    h = get_multi_layer_state(text, SELECTED_LAYERS).reshape(1, -1)
    h_scaled = scaler.transform(h)
    token_type = probe.predict(h_scaled)[0]
    probability = probe.predict_proba(h_scaled)[0, 1]
    return int(token_type), float(probability)

def analyze_candidate_tokens(
    prompt: str,
    top_k: int = 10,
    verbose: bool = False
) -> Dict:
    """
    Get top-K candidate next tokens and classify each as CODE or LANGUAGE.
    Returns majority vote and detailed breakdown.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :].cpu().numpy()

    probs = softmax(logits)
    top_indices = np.argsort(probs)[-top_k:][::-1]

    candidates = []
    code_votes = 0
    lang_votes = 0

    for idx in top_indices:
        token = tokenizer.decode([idx])
        prob = probs[idx]

        # Classify prompt + candidate token
        completion = prompt + token
        token_type, type_prob = classify_token_type(completion)

        candidates.append({
            'token': token,
            'prob': prob,
            'type': 'CODE' if token_type == 1 else 'LANGUAGE',
            'type_prob': type_prob
        })

        if token_type == 1:
            code_votes += prob  # Weight by token probability
        else:
            lang_votes += prob

    # Decision based on probability-weighted votes
    is_code_uncertainty = code_votes > lang_votes

    if verbose:
        print(f"\nCandidate analysis:")
        for c in candidates:
            print(f"  '{c['token']}' (p={c['prob']:.3f}) → {c['type']} (conf={c['type_prob']:.3f})")
        print(f"\nVotes: CODE={code_votes:.3f}, LANGUAGE={lang_votes:.3f}")
        print(f"Decision: {'CODE uncertainty' if is_code_uncertainty else 'LANGUAGE uncertainty'}")

    return {
        'candidates': candidates,
        'code_votes': code_votes,
        'lang_votes': lang_votes,
        'is_code_uncertainty': is_code_uncertainty,
        'confidence': max(code_votes, lang_votes) / (code_votes + lang_votes)
    }

print("✅ Contrastive helper functions ready")

✅ Contrastive helper functions ready


In [9]:
# Cell 9: Contrastive generation function

def generate_with_contrastive_probe(
    prompt: str,
    entropy_threshold: float = 3.0,
    top_k_candidates: int = 10,
    max_tokens: int = 50,
    verbose: bool = True
) -> Dict:
    """
    Contrastive entropy-driven generation.

    At each step:
    1. Generate next token
    2. Compute entropy H
    3. If H > threshold:
       - Get top-K candidate next tokens
       - Classify each "prompt + candidate" as CODE or LANGUAGE
       - Weighted vote: If majority CODE → STOP
       - If majority LANGUAGE → Continue
    4. If H ≤ threshold: Continue (confident)
    """
    current_text = prompt
    generated_tokens = []
    entropy_trace = []
    stop_reason = None
    stop_info = {}

    if verbose:
        print(f"\n{'='*80}")
        print(f"Prompt: '{prompt}'")
        print(f"Entropy threshold: {entropy_threshold:.1f} bits")
        print(f"{'='*80}")

    for step in range(max_tokens):
        inputs = tokenizer(current_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits[0, -1, :].cpu().numpy()

        probs = softmax(logits)
        H = entropy_from_probs(probs)
        entropy_trace.append(H)

        next_token_id = np.argmax(probs)
        next_token = tokenizer.decode([next_token_id])

        if verbose:
            print(f"\nStep {step + 1}: '{next_token}' H={H:.2f}")

        # Check entropy
        if H > entropy_threshold:
            if verbose:
                print(f"  ⚠️  HIGH ENTROPY - analyzing candidates...")

            # Analyze what the model is uncertain about
            analysis = analyze_candidate_tokens(
                current_text,
                top_k=top_k_candidates,
                verbose=verbose
            )

            if analysis['is_code_uncertainty']:
                if verbose:
                    print(f"  ❗ CODE UNCERTAINTY - STOPPING!")
                    print(f"  → Model is uncertain about code-specific tokens")
                stop_reason = "code_uncertainty"
                stop_info = {
                    'step': step,
                    'entropy': H,
                    'candidates': analysis['candidates'],
                    'code_votes': analysis['code_votes'],
                    'lang_votes': analysis['lang_votes'],
                    'confidence': analysis['confidence']
                }
                break
            else:
                if verbose:
                    print(f"  ✓ LANGUAGE uncertainty - continuing")

        generated_tokens.append(next_token)
        current_text += next_token

        if next_token_id == tokenizer.eos_token_id:
            stop_reason = "eos"
            break

    if stop_reason is None:
        stop_reason = "max_tokens"

    if verbose:
        print(f"\n{'='*80}")
        print(f"Stop: {stop_reason}")
        print(f"Generated: '{current_text}'")
        print(f"{'='*80}")

    return {
        'prompt': prompt,
        'generated_text': ''.join(generated_tokens),
        'full_text': current_text,
        'entropy_trace': entropy_trace,
        'stop_reason': stop_reason,
        'stop_info': stop_info,
        'num_steps': len(generated_tokens)
    }

print("✅ Contrastive generation function ready")

✅ Contrastive generation function ready


# Part 5: Test the System

In [10]:
# Cell 10: Quick demo

print("\n" + "="*80)
print("DEMO: Testing Contrastive Probe")
print("="*80)

test_prompts = [
    "The authentication is done using",  # Should STOP (code uncertainty)
    "The authentication is",              # Should CONTINUE (language uncertainty)
]

for test_prompt in test_prompts:
    result = generate_with_contrastive_probe(
        test_prompt,
        entropy_threshold=3.0,
        top_k_candidates=10,
        max_tokens=5,
        verbose=True
    )
    print("\n" + "-"*80 + "\n")


DEMO: Testing Contrastive Probe

Prompt: 'The authentication is done using'
Entropy threshold: 3.0 bits

Step 1: 'the' H=5.88
  ⚠️  HIGH ENTROPY - analyzing candidates...

Candidate analysis:
  'the' (p=0.235) → LANGUAGE (conf=0.000)
  'a' (p=0.158) → LANGUAGE (conf=0.000)
  'O' (p=0.053) → LANGUAGE (conf=0.000)
  'an' (p=0.046) → LANGUAGE (conf=0.000)
  '[' (p=0.040) → LANGUAGE (conf=0.000)
  '`' (p=0.029) → LANGUAGE (conf=0.000)
  'J' (p=0.025) → LANGUAGE (conf=0.006)
  'Open' (p=0.015) → LANGUAGE (conf=0.005)
  'HTTP' (p=0.014) → CODE (conf=0.892)
  'Spring' (p=0.009) → CODE (conf=1.000)

Votes: CODE=0.023, LANGUAGE=0.601
Decision: LANGUAGE uncertainty
  ✓ LANGUAGE uncertainty - continuing

Step 2: '`' H=8.73
  ⚠️  HIGH ENTROPY - analyzing candidates...

Candidate analysis:
  '`' (p=0.070) → LANGUAGE (conf=0.000)
  'following' (p=0.062) → LANGUAGE (conf=0.000)
  'O' (p=0.038) → LANGUAGE (conf=0.000)
  '[' (p=0.033) → LANGUAGE (conf=0.000)
  'J' (p=0.018) → LANGUAGE (conf=0.003)
  '

In [11]:
# Cell 11: Comprehensive test dataset

CODE_TEST_CASES = [
    {'prompt': 'The authentication is done using', 'expected': 1, 'category': 'auth_method'},
    {'prompt': 'We import', 'expected': 1, 'category': 'import_library'},
    {'prompt': 'The database is', 'expected': 1, 'category': 'database_type'},
    {'prompt': 'The frontend uses', 'expected': 1, 'category': 'frontend_framework'},
    {'prompt': 'Password hashing uses', 'expected': 1, 'category': 'auth_hash'},
    {'prompt': 'We use model.', 'expected': 1, 'category': 'ml_method'},
    {'prompt': 'SELECT password FROM', 'expected': 1, 'category': 'database_table'},
    {'prompt': 'The API uses', 'expected': 1, 'category': 'web_framework'},
    {'prompt': 'State management is', 'expected': 1, 'category': 'frontend_state'},
    {'prompt': 'Tokens are generated with', 'expected': 1, 'category': 'auth_token'},
]

LANGUAGE_TEST_CASES = [
    {'prompt': 'The authentication is', 'expected': 0, 'category': 'connector_is'},
    {'prompt': 'Verification is done', 'expected': 0, 'category': 'connector_done'},
    {'prompt': 'The user id is', 'expected': 0, 'category': 'connector_is'},
    {'prompt': 'Processing is handled', 'expected': 0, 'category': 'connector_handled'},
    {'prompt': 'Data is stored', 'expected': 0, 'category': 'connector_stored'},
    {'prompt': 'The code works by', 'expected': 0, 'category': 'explanation_process'},
    {'prompt': 'The function is responsible for', 'expected': 0, 'category': 'explanation_responsibility'},
    {'prompt': 'To implement authentication,', 'expected': 0, 'category': 'instruction_to'},
    {'prompt': 'First, you need to', 'expected': 0, 'category': 'instruction_first'},
    {'prompt': 'The authentication system', 'expected': 0, 'category': 'explanation_system'},
]

ALL_TEST_CASES = CODE_TEST_CASES + LANGUAGE_TEST_CASES

print(f"\n{'='*80}")
print(f"TEST DATASET")
print(f"{'='*80}")
print(f"\nTotal tests: {len(ALL_TEST_CASES)}")
print(f"  CODE tests (should stop): {len(CODE_TEST_CASES)}")
print(f"  LANGUAGE tests (should continue): {len(LANGUAGE_TEST_CASES)}")
print(f"{'='*80}")

# Run tests
print(f"\nRunning {len(ALL_TEST_CASES)} test cases...\n")

test_results = []
for test_case in tqdm(ALL_TEST_CASES, desc="Testing"):
    result = generate_with_contrastive_probe(
        test_case['prompt'],
        entropy_threshold=3.0,
        top_k_candidates=10,
        max_tokens=5,
        verbose=False
    )

    predicted = 1 if result['stop_reason'] == 'code_uncertainty' else 0
    expected = test_case['expected']
    correct = predicted == expected

    test_results.append({
        'prompt': test_case['prompt'],
        'category': test_case['category'],
        'expected': expected,
        'predicted': predicted,
        'correct': correct,
        'stop_reason': result['stop_reason'],
        'max_entropy': np.max(result['entropy_trace']) if result['entropy_trace'] else 0,
    })

print(f"\n✅ Testing complete!")


TEST DATASET

Total tests: 20
  CODE tests (should stop): 10
  LANGUAGE tests (should continue): 10

Running 20 test cases...



Testing:   0%|          | 0/20 [00:00<?, ?it/s]


✅ Testing complete!


In [12]:
# Cell 12: Results analysis

df_results = pd.DataFrame(test_results)

print("\n" + "="*80)
print("CONTRASTIVE PROBE TEST RESULTS")
print("="*80)

overall_accuracy = df_results['correct'].mean()
print(f"\n📊 OVERALL ACCURACY: {overall_accuracy:.1%}")

print(f"\n📈 BY CLASS:")
for expected_val in [1, 0]:
    class_name = "CODE (should stop)" if expected_val == 1 else "LANGUAGE (should continue)"
    subset = df_results[df_results['expected'] == expected_val]
    accuracy = subset['correct'].mean()
    correct = subset['correct'].sum()
    total = len(subset)
    print(f"\n   {class_name}")
    print(f"      Correct: {correct}/{total}")
    print(f"      Accuracy: {accuracy:.1%}")

# Confusion matrix
tp = len(df_results[(df_results['expected']==1) & (df_results['predicted']==1)])
fp = len(df_results[(df_results['expected']==0) & (df_results['predicted']==1)])
fn = len(df_results[(df_results['expected']==1) & (df_results['predicted']==0)])
tn = len(df_results[(df_results['expected']==0) & (df_results['predicted']==0)])

print(f"\n🎯 CONFUSION MATRIX")
print(f"                    Predicted LANGUAGE    Predicted CODE")
print(f"True LANGUAGE             {tn:<12}      {fp:<12}")
print(f"True CODE                 {fn:<12}      {tp:<12}")

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n📐 METRICS (CODE class)")
print(f"   Precision: {precision:.1%}")
print(f"   Recall: {recall:.1%}")
print(f"   F1-Score: {f1:.1%}")

# Errors
errors = df_results[~df_results['correct']]
if len(errors) > 0:
    print(f"\n❌ ERRORS ({len(errors)}/{len(df_results)}):")
    for _, error in errors.iterrows():
        exp = "CODE" if error['expected'] == 1 else "LANGUAGE"
        pred = "CODE" if error['predicted'] == 1 else "LANGUAGE"
        print(f"   '{error['prompt'][:50]}...' Expected: {exp}, Predicted: {pred}")
else:
    print(f"\n🎉 PERFECT! No errors!")

print(f"\n" + "="*80)


CONTRASTIVE PROBE TEST RESULTS

📊 OVERALL ACCURACY: 55.0%

📈 BY CLASS:

   CODE (should stop)
      Correct: 3/10
      Accuracy: 30.0%

   LANGUAGE (should continue)
      Correct: 8/10
      Accuracy: 80.0%

🎯 CONFUSION MATRIX
                    Predicted LANGUAGE    Predicted CODE
True LANGUAGE             8                 2           
True CODE                 7                 3           

📐 METRICS (CODE class)
   Precision: 60.0%
   Recall: 30.0%
   F1-Score: 40.0%

❌ ERRORS (9/20):
   'The authentication is done using...' Expected: CODE, Predicted: LANGUAGE
   'The frontend uses...' Expected: CODE, Predicted: LANGUAGE
   'Password hashing uses...' Expected: CODE, Predicted: LANGUAGE
   'We use model....' Expected: CODE, Predicted: LANGUAGE
   'The API uses...' Expected: CODE, Predicted: LANGUAGE
   'State management is...' Expected: CODE, Predicted: LANGUAGE
   'Tokens are generated with...' Expected: CODE, Predicted: LANGUAGE
   'The authentication is...' Expected: LANGUAG

# Summary

## What We Did

1. **Contrastive Training Data**: Instead of training on static prompts, we trained on "prompt + candidate token" pairs
   - "The authentication is done using **JWT**" → CODE
   - "The authentication is **a**" → LANGUAGE

2. **Contrastive Classification**: At generation time, when entropy is high:
   - Get top-K candidate next tokens
   - Classify each "current_text + candidate"
   - Weighted vote based on token probabilities
   - If majority vote is CODE → stop and ask
   - If majority vote is LANGUAGE → continue

3. **Key Advantage**: We're now classifying **what the model is uncertain about** rather than **where the model is** in generation.

This should fix the distribution shift problem and dramatically improve LANGUAGE test accuracy!